# Step 4: Object Storage with S3

Thus far, the "lake" has consisted of a folder on local disk. In production, data lakes typically reside on **object storage** (S3, Azure Blob, GCS, etc.): storage and compute operate as separate services, scale independently, and can be shared by multiple engines simultaneously.

[moto](https://github.com/getmoto/moto) is a Python package that runs a local server speaking the same API as AWS S3. It runs as a separate process directly within the Codespace and requires no cloud account. Moto keeps all data in memory, so a bucket and its files are gone once the server stops; for a teaching environment that is sufficient, since the files can be uploaded again at any time.

**Before running this notebook**, start the S3 server from a terminal:

```bash
bash scripts/start_s3.sh
```

Codespaces forwards port **9000** automatically. Open the "Ports" tab, select the link, and append `/moto-api/` to the URL to reach moto's dashboard, which lists the buckets and objects the server currently holds.

In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

## Creating a Bucket and Uploading the Parquet Files

A bucket is the S3 equivalent of a top-level folder. The partitioned Parquet files from `lake/verkauf/` are uploaded to a bucket named `datalake`, preserving the `jahr=.../monat=.../data.parquet` key structure — partitioning functions identically on object storage.

In [ ]:
import boto3

S3_ENDPOINT = "localhost:9000"
S3_ACCESS_KEY = "test"   # moto accepts any credentials
S3_SECRET_KEY = "test"
BUCKET = "datalake"

client = boto3.client(
    "s3",
    endpoint_url=f"http://{S3_ENDPOINT}",
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name="us-east-1",
)

existing = [b["Name"] for b in client.list_buckets()["Buckets"]]
if BUCKET not in existing:
    client.create_bucket(Bucket=BUCKET)
    print(f"Created bucket '{BUCKET}'")
else:
    print(f"Bucket '{BUCKET}' already exists")

In [ ]:
local_root = Path("lake/verkauf")
uploaded = 0

for file in local_root.rglob("*.parquet"):
    object_key = "verkauf/" + file.relative_to(local_root).as_posix()
    client.upload_file(str(file), BUCKET, object_key)
    uploaded += 1

print(f"Uploaded {uploaded} files to s3://{BUCKET}/verkauf/")

Opening moto's dashboard (port 9000, path `/moto-api/`) at this point shows the `datalake` bucket and its objects, whose keys carry the same `jahr=.../monat=.../` structure, visible directly in the browser.

## Querying S3 Directly from DuckDB

DuckDB's `httpfs` extension implements the S3 API. `CREATE SECRET` registers the S3 credentials and endpoint once per session; thereafter, `s3://...` paths function exactly like local paths.

In [ ]:
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs")
con.sql("LOAD httpfs")

con.sql(f"""
    CREATE SECRET s3_secret (
        TYPE S3,
        KEY_ID '{S3_ACCESS_KEY}',
        SECRET '{S3_SECRET_KEY}',
        ENDPOINT '{S3_ENDPOINT}',
        URL_STYLE 'path',
        USE_SSL false,
        REGION 'us-east-1'
    )
""")
print("Secret registered.")

In [ ]:
con.sql("""
    SELECT region, SUM(revenue) AS total_revenue
    FROM 's3://datalake/verkauf/**/*.parquet'
    GROUP BY region
    ORDER BY total_revenue DESC
""").show()

## Key Takeaway

Comparing this query to the one in `03_duckdb_queries.ipynb` shows that it is identical except for the path — `lake/verkauf/...` has become `s3://datalake/verkauf/...`. DuckDB, the compute/query engine, remains unchanged; only the underlying storage layer differs.

Concretely, storage and compute are two separate processes here, communicating only over the network:

- **Storage is the S3 server (moto)** — a separate process (started beforehand via `bash scripts/start_s3.sh`), listening on `localhost:9000` for the S3 API. It holds the actual Parquet bytes inside the `datalake` bucket and does nothing but serve them on request.
- **Compute is DuckDB** — running inside this notebook's own Python kernel, a different process from the S3 server entirely. It parses the SQL, plans the query, applies predicate pushdown and column pruning, and performs the aggregation.
- **The link between them** is the `httpfs` extension, which speaks the S3 protocol over HTTP to the endpoint registered in `CREATE SECRET`. DuckDB requests only the bytes it needs; the S3 server has no awareness of SQL, query plans, or aggregation.

This is what makes the separation real rather than conceptual: killing the notebook's kernel would leave the S3 server and the data untouched, and stopping the S3 server would leave DuckDB fully intact as a query engine, just with nothing left to query. The same engine can query a local folder, an S3 bucket, or, as in production, a cloud data lake, with no change required to the SQL.